# הורדת Setup-SochenChidushim-1.bin ל-Google Drive
הקובץ שוקל כ-22.6GB ומורד ב-11 חלקים של 2GB לתיקייה `sochen` ב-Drive.

1. הרץ את שני התאים הראשונים (חיבור ל-Drive והגדרות).
2. הרץ את תא `run()`. **השאר את הלשונית פתוחה.**
3. אם אין מקום ב-Drive - התא מחכה. רוקן מקום (כולל הפח) והוא ממשיך לבד.
4. אם Colab התנתק: חבר מחדש והרץ שוב את אותם תאים - חלקים שהסתיימו לא יורדו שוב.
5. בסוף הרץ את תא האימות.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, hashlib, shutil, requests

URL    = "https://huggingface.co/datasets/NOAM212819/torat-melech/resolve/main/sochen/Setup-SochenChidushim-1.bin?download=true"
TOTAL  = 22608727854                      # גודל הקובץ המלא בבתים
SHA256 = "0aab79123116fcb59d8fde0ac49ae47ef5348b258e1f6c6f0cb9c513e20d6e10"
PART   = 2 * 1024**3                      # גודל כל חלק: 2GB (בסך הכול 11 חלקים)
OUT    = "/content/drive/MyDrive/sochen"  # התיקייה ב-Drive
NAME   = "Setup-SochenChidushim-1.bin"
N      = -(-TOTAL // PART)

def part_path(i): return f"{OUT}/{NAME}.part{i:03d}"
def part_len(i):  return min(PART, TOTAL - i * PART)
def gb(x):        return f"{x/1024**3:.2f}GB"

def download_part(i):
    path, want = part_path(i), part_len(i)
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have > want:
        os.remove(path); have = 0
    if have == want:
        return
    start, end = i * PART + have, i * PART + want - 1
    t0, last, got = time.time(), time.time(), 0
    with requests.get(URL, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=60) as r:
        r.raise_for_status()
        if r.status_code != 206:
            raise RuntimeError(f"השרת החזיר {r.status_code} במקום 206")
        with open(path, "ab") as f:
            for chunk in r.iter_content(8 << 20):
                f.write(chunk); got += len(chunk)
                if time.time() - last > 30:
                    last = time.time()
                    print(f"   חלק {i}: {gb(have+got)}/{gb(want)}  ({got/(time.time()-t0)/1e6:.0f} MB/s)")
            f.flush(); os.fsync(f.fileno())
    if os.path.getsize(path) != want:
        raise OSError(f"גודל לא תקין: {os.path.getsize(path)} במקום {want}")

def run():
    os.makedirs(OUT, exist_ok=True)
    i, net_fail = 0, 0
    while i < N:
        if os.path.exists(part_path(i)) and os.path.getsize(part_path(i)) == part_len(i):
            print(f"חלק {i}/{N-1} כבר קיים - מדלג"); i += 1; continue
        print(f"מוריד חלק {i}/{N-1} ({gb(part_len(i))}) ...")
        try:
            download_part(i)
            print(f"✔ חלק {i} הסתיים"); i += 1; net_fail = 0
        except requests.RequestException as e:      # בעיית רשת - ממשיכים מאותה נקודה
            net_fail += 1
            print(f"⚠ בעיית רשת ({e.__class__.__name__}), ניסיון {net_fail}/30, ממשיך בעוד 15 שניות")
            if net_fail >= 30: raise
            time.sleep(15)
        except OSError as e:                         # כמעט תמיד: אין מקום ב-Drive
            try: os.remove(part_path(i))
            except OSError: pass
            print(f"⛔ כנראה אין מקום פנוי ב-Drive ({e}).")
            print("   רוקן מקום (וגם את הפח ב-drive.google.com/drive/trash) - ההורדה תמשיך לבד בתוך דקה. "
                  "אל תסגור את הלשונית.")
            time.sleep(60)
    print("🎉 כל החלקים הורדו. הרץ את תא האימות.")

In [ ]:
run()

In [ ]:
# אימות: בודק גדלים ו-SHA256 של הקובץ השלם (קורא את החלקים לפי הסדר, לא צריך מקום נוסף)
h = hashlib.sha256()
for i in range(N):
    p = part_path(i)
    assert os.path.exists(p) and os.path.getsize(p) == part_len(i), f"חסר או פגום: חלק {i}"
    with open(p, "rb") as f:
        while chunk := f.read(16 << 20):
            h.update(chunk)
    print("נבדק חלק", i)
print("SHA256:", h.hexdigest())
print("✅ הקובץ תקין" if h.hexdigest() == SHA256 else "❌ ה-SHA256 לא תואם - הורד מחדש את החלק הפגום")

## חיבור החלקים לקובץ אחד
החיבור צריך עוד 22.6GB פנויים, לכן הכי נוח לעשות אותו במחשב שלך אחרי שהורדת את החלקים מה-Drive:

- Windows (CMD, בתיקייה עם החלקים): `copy /b Setup-SochenChidushim-1.bin.part* Setup-SochenChidushim-1.bin`
- Mac/Linux: `cat Setup-SochenChidushim-1.bin.part* > Setup-SochenChidushim-1.bin`